# Logit Lens Decomposition

Decompose logit lens predictions into component contributions:
per-layer predictions, attn vs MLP attribution, and entropy trajectories.

## Why This Matters

Composition logit lens decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_lens_decomposition import (
    per_layer_logit_lens, component_logit_contribution,
    prediction_change_attribution, logit_lens_entropy_trajectory,
    logit_lens_decomposition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Per-Layer Logit Lens

What would the model predict if decoding at each layer?

In [ ]:
result = per_layer_logit_lens(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: top={p['top_token']} (p={p['top_prob']:.4f}), H={p['entropy']:.4f}")

## Component Logit Contribution

Attention vs MLP contribution to the logit lens at each layer.

In [ ]:
for layer in range(4):
    result = component_logit_contribution(model, tokens, layer=layer, position=-1)
    attn_bar = '█' * int(result['attn_fraction'] * 20)
    mlp_bar = '█' * int(result['mlp_fraction'] * 20)
    print(f"  Layer {layer}: attn={result['attn_fraction']:.2f} {attn_bar}")
    print(f"           mlp ={result['mlp_fraction']:.2f} {mlp_bar}")

## Prediction Change Attribution

Which components drive prediction shifts?

In [ ]:
result = prediction_change_attribution(model, tokens, position=-1)
print(f"Total changes: {result['n_prediction_changes']}, final: {result['final_token']}")
for p in result['per_layer']:
    flag = ' ← CHANGED' if p['changed_prediction'] else ''
    print(f"  Layer {p['layer']}: top={p['top_token']}, "
          f"attn={p['attn_logit_for_top']:+.4f}, mlp={p['mlp_logit_for_top']:+.4f}{flag}")

## Entropy Trajectory

Are predictions getting sharper through layers?

In [ ]:
result = logit_lens_entropy_trajectory(model, tokens, position=-1)
flag = ' [SHARPENING]' if result['is_sharpening'] else ''
print(f"Entropy reduction: {result['entropy_reduction']:.4f}{flag}")
for p in result['per_layer']:
    bar = '█' * int(p['entropy'])
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f}, top_p={p['top_prob']:.4f} {bar}")

## Decomposition Summary

In [ ]:
result = logit_lens_decomposition_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")